# RoBERTa Fine-Tuning for Word Ladder

Fine-tune RoBERTa to predict whether a **candidate** word is the correct next step on a shortest path from **current** to **target**.

**Input format:** `current [SEP] target` + `candidate` (sentence-pair classification)

**Output:** Binary label (1 = correct next step, 0 = wrong)

**Data:** `data/training/wordladder_english5_*.csv` from notebook 05

**Dependencies:** `pip install transformers torch pandas tqdm accelerate`

## 1. Setup and imports

In [1]:
# Ensure accelerate is installed (required by Trainer). Run this cell FIRST if you get ImportError.
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "accelerate>=1.1.0", "-q"])
import accelerate
print(f"accelerate {accelerate.__version__} OK")

c:\Users\doric\Documents\GitHub\word-ladder\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


accelerate 1.13.0 OK


In [2]:
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

BASE = Path("../data")
TRAINING = BASE / "training"
MODEL_NAME = "roberta-base"
MAX_LENGTH = 64  # plenty for "word1 [SEP] word2" + "word3"
BATCH_SIZE = 32
EPOCHS = 3
SEED = 42

torch.manual_seed(SEED)
print(f"PyTorch: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

PyTorch: 2.10.0+cpu
Device: cpu


## 2. Load the training data

Read the CSVs. Each row has `text_a` (current [SEP] target), `text_b` (candidate), and `label` (0 or 1).

In [3]:
train_df = pd.read_csv(TRAINING / "wordladder_english5_train.csv")
val_df = pd.read_csv(TRAINING / "wordladder_english5_val.csv")
test_df = pd.read_csv(TRAINING / "wordladder_english5_test.csv")

print(f"Train: {len(train_df)} examples")
print(f"Val: {len(val_df)} examples")
print(f"Test: {len(test_df)} examples")
print(f"Class balance (train): {train_df['label'].mean():.2%} positive")
print(f"\nSample row:")
print(train_df.iloc[0].to_dict())

Train: 13444 examples
Val: 785 examples
Test: 771 examples
Class balance (train): 25.18% positive

Sample row:
{'text_a': 'cruse [SEP] gnash', 'text_b': 'crude', 'label': 0}


## 3. Load tokenizer and tokenize

For BERT sentence-pair classification we encode: `[CLS] text_a [SEP] text_b [SEP]`

The tokenizer returns `input_ids` and `attention_mask` which we'll pass to the model.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_pair(text_a: str, text_b: str):
    """Tokenize (text_a, text_b) for BERT sentence-pair classification."""
    return tokenizer(
        text_a,
        text_b,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors=None,  # return lists, not tensors
    )


def tokenize_df(df: pd.DataFrame):
    """Tokenize all rows in a dataframe."""
    encodings = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Tokenizing"):
        enc = tokenize_pair(str(row["text_a"]), str(row["text_b"]))
        encodings.append(enc)
    return encodings


train_enc = tokenize_df(train_df)
val_enc = tokenize_df(val_df)
test_enc = tokenize_df(test_df)
print(f"Tokenized {len(train_enc)} train, {len(val_enc)} val, {len(test_enc)} test")

Tokenizing: 100%|██████████| 771/771 [00:00<00:00, 2996.93it/s]

Tokenized 13444 train, 785 val, 771 test


## 4. Create PyTorch Dataset

Wrap tokenized data + labels into a Dataset that returns dicts of tensors.

In [5]:
class WordLadderDataset(Dataset):
    """Dataset of (input_ids, attention_mask, labels) for BERT."""

    def __init__(self, encodings: list, labels: list):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v) for k, v in self.encodings[idx].items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_dataset = WordLadderDataset(train_enc, train_df["label"].tolist())
val_dataset = WordLadderDataset(val_enc, val_df["label"].tolist())
test_dataset = WordLadderDataset(test_enc, test_df["label"].tolist())

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Sample batch keys: {list(train_dataset[0].keys())}")

Train dataset: 13444 examples
Sample batch keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']


## 5. Load model and set up Trainer

We use `AutoModelForSequenceClassification` with 2 labels (binary). The HuggingFace `Trainer` handles training loop, logging, and evaluation.

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir="../outputs/bert_wordladder",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    seed=SEED,
)


def compute_metrics(eval_pred):
    """Compute accuracy for the Trainer."""
    import numpy as np
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = (preds == labels).mean()
    return {"accuracy": acc}


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print(f"Model: {MODEL_NAME}")
print(f"Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1657.42it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider tr

Model: bert-base-uncased
Epochs: 3, Batch size: 32


## 6. Train

In [8]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.507553,0.529531,0.774522
2,0.494332,0.525537,0.755414
3,0.390879,0.561833,0.757962


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.69it/s]
c:\Users\doric\Documents\GitHub\word-ladder\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.98it/s]
c:\Users\doric\Documents\GitHub\word-ladder\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.28it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert

TrainOutput(global_step=1263, training_loss=0.45724175076397844, metrics={'train_runtime': 7748.178, 'train_samples_per_second': 5.205, 'train_steps_per_second': 0.163, 'total_flos': 1350680602690560.0, 'train_loss': 0.45724175076397844, 'epoch': 3.0})

## 7. Evaluate on validation and test

In [10]:
# Manual evaluation (avoids callback error when running cell separately)
from torch.utils.data import DataLoader

def eval_dataset(model, dataset, batch_size=32):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels")
            out = model(**{k: v.to(model.device) for k, v in batch.items()})
            preds = out.logits.argmax(dim=1)
            correct += (preds.cpu() == labels).sum().item()
            total += len(labels)
    return {"accuracy": correct / total}

val_result = eval_dataset(trainer.model, val_dataset)
test_result = eval_dataset(trainer.model, test_dataset)

print("\n=== Validation ===")
print(f"  accuracy: {val_result['accuracy']:.4f}")

print("\n=== Test ===")
print(f"  accuracy: {test_result['accuracy']:.4f}")


=== Validation ===
  accuracy: 0.7745

=== Test ===
  accuracy: 0.7951


## 8. Save the model

Save to disk so we can load it later for inference.

In [11]:
SAVE_PATH = Path("../models/bert_wordladder_5letter")
SAVE_PATH.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(SAVE_PATH))
tokenizer.save_pretrained(str(SAVE_PATH))

print(f"Model and tokenizer saved to {SAVE_PATH}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.87it/s]

Model and tokenizer saved to ..\models\bert_wordladder_5letter


## 9. Inference helper

Given (current, target) and a list of candidate words, score each candidate and return the best one (or ranked list).

In [13]:
def score_candidates(model, tokenizer, current: str, target: str, candidates: list, device=None):
    """
    Score each candidate. Returns list of (candidate, score) sorted by score descending.
    Score = probability of label 1 (correct next step).
    """
    if device is None:
        device = next(model.parameters()).device

    text_a = f"{current} [SEP] {target}"
    scores = []

    model.eval()
    with torch.no_grad():
        for cand in candidates:
            enc = tokenizer(
                text_a,
                cand,
                truncation=True,
                max_length=MAX_LENGTH,
                padding="max_length",
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            out = model(**enc)
            probs = torch.softmax(out.logits, dim=1)
            score = probs[0, 1].item()  # P(label=1)
            scores.append((cand, score))

    return sorted(scores, key=lambda x: -x[1])


# Quick test with 5-letter words (in-distribution)
import string

def one_letter_neighbors(w: str, vocab: set) -> set:
    return {w[:i]+c+w[i+1:] for i in range(len(w)) for c in string.ascii_lowercase if c!=w[i]} & vocab

vocab = {w.strip() for w in (BASE / "islands" / "english_5_largest_island.txt").read_text(encoding="utf-8").splitlines() if w.strip()}
pos = val_df[val_df["label"]==1].iloc[0]
parts = pos["text_a"].split(" [SEP] ")
test_current, test_target = parts[0], parts[1]
correct_next = pos["text_b"]
test_candidates = list(one_letter_neighbors(test_current, vocab))[:8]
if correct_next not in test_candidates:
    test_candidates.append(correct_next)
ranked = score_candidates(model, tokenizer, test_current, test_target, test_candidates)
print(f"Current: {test_current}, Target: {test_target}")
print(f"Correct next (from data): {correct_next}")
print("Ranked candidates:")
for cand, score in ranked:
    mark = " <-- correct" if cand == correct_next else ""
    print(f"  {cand}: {score:.3f}{mark}")

Current: saned, Target: scrip
Correct next (from data): sayed
Ranked candidates:
  sanes: 0.234
  sayed: 0.210 <-- correct
  saner: 0.201
  sabed: 0.196
  paned: 0.187
  caned: 0.179
  maned: 0.172
  waned: 0.156
  saved: 0.144
